In [1]:
!pip install sentence-transformers faiss-cpu transformers pandas pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 66.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 19.4 MB/s eta 0:00:00


In [2]:
# upload my law material paragraphs (relevant for questions)
from google.colab import files

uploaded = files.upload()

Saving BAO.pdf to BAO.pdf
Saving UStG1994.pdf to UStG1994.pdf
Saving KStG1988.pdf to KStG1988.pdf
Saving EStG1988.pdf to EStG1988.pdf
Saving dataset_clean.csv to dataset_clean.csv


In [3]:
import pandas as pd
from pypdf import PdfReader

# load cleaned dataset from csv file into dataframe
dataset = pd.read_csv("dataset_clean.csv")


# list of tax law pdf files to extract text from
pdf_files = [
    "BAO.pdf",
    "UStG1994.pdf",
    "KStG1988.pdf",
    "EStG1988.pdf"
]


# function to extract text from a single pdf file
def extract_text(pdf):
    reader = PdfReader(pdf)  # open pdf file
    text = ""
    for page in reader.pages:  # loop through all pages
        text += page.extract_text() + "\n"  # add page text to string
    return text  # return full extracted text


law_text = ""  # create empty string to store combined law text


# extract and combine text from all pdf law files
for pdf in pdf_files:
    law_text += extract_text(pdf)


# check total length of extracted legal text (number of characters)
len(law_text)

409012

In [4]:
# function to split long text into smaller word-based chunks
def chunk_text(text, chunk_size=500):
    words = text.split()  # split text into individual words
    chunks = []  # create empty list to store chunks

    for i in range(0, len(words), chunk_size):  # iterate in steps of chunk_size
        chunk = " ".join(words[i:i+chunk_size])  # join words into chunk
        chunks.append(chunk)  # add chunk to list

    return chunks  # return list of text chunks


# apply chunking function to extracted law text
chunks = chunk_text(law_text)

# check how many chunks were created
len(chunks)

114

In [5]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# load multilingual embedding model
embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# convert text chunks into vector embeddings
chunk_embeddings = embedder.encode(chunks, show_progress_bar=True)

# get embedding vector size
dimension = chunk_embeddings.shape[1]

# create faiss similarity search index
index = faiss.IndexFlatL2(dimension)

# store embeddings inside index for retrieval
index.add(np.array(chunk_embeddings))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

In [32]:
from transformers import pipeline

# create text-to-text generation pipeline with flan-t5-small model
generator = pipeline(
    "text2text-generation",   # task type: instruction text generation
    model="google/flan-t5-small"  # lightweight instruction-tuned model
)

KeyError: "Unknown task text2text-generation, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'image-to-image', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'question-answering', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'visual-question-answering', 'vqa', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection', 'translation_XX_to_YY']"

In [14]:
def retrieve_context(question, k=2, max_chars=1200):
    # create embedding for the input question
    question_embedding = embedder.encode([question])

    # search faiss index for top-k most similar chunks
    distances, indices = index.search(
        np.array(question_embedding),
        k
    )

    context = ""

    # collect retrieved text chunks
    for i in indices[0]:
        context += chunks[i] + "\n"

    # limit context length to avoid model input overflow
    return context[:max_chars]

In [31]:
def rag_answer(question):

    # retrieve relevant law text chunks using semantic search
    context = retrieve_context(question, k=2, max_chars=600)

    # build prompt with question + retrieved legal context
    prompt = f"""
Beantworte die steuerrechtliche Frage. Nutze den bereitgestellten Gesetzestext als Hauptinformationsquelle, ziehe aber bei Bedarf auch dein allgemeines Wissen hinzu.

Frage:
{question}

Gesetzestext:
{context}

Antwort:
"""

    # generate answer using flan-t5 model
    result = generator(
        prompt,
        max_new_tokens=40,   # limit answer length
        do_sample=False      # deterministic output
    )

    # return cleaned generated answer text
    return result[0]["generated_text"].strip()

In [28]:
# retrieve most relevant legal text chunks for the test question
context = retrieve_context(test_question)

# print heading for readability in console output
print("\nGefundener Kontext:")

# print first 1000 characters of retrieved context (preview only)
print(context[:1000])


Gefundener Kontext:
Einkünften aus dieser Betätigung oder diesem Betrieb frühestmöglich zu verrechnen. (Anm.: Abs. 2b aufgehoben durch BGBl. I Nr. 13/2014 ) 4 . (3) Der Einkommensteuer unterliegen nur: 1 . 1. Einkünfte aus Land- und Forstwirtschaft ( § 21 ), 2 . 2. Einkünfte aus selbständiger Arbeit ( § 22 ), 3 . 3. Einkünfte aus Gewerbebetrieb ( § 23 ), 4 . 4. Einkünfte aus nichtselbständiger Arbeit ( § 25 ), 5 . 5. Einkünfte aus Kapitalvermögen ( § 27 ), 6 . 6. Einkünfte aus Vermietung und Verpachtung ( § 28 ), 7 . 7. sonstige Einkünfte im Sinne des § 29 . 5 . (4) Einkünfte im Sinne des Abs. 3 sind: 1 . 1. Der Gewinn ( §§ 4 bis 14) bei Land- und Forstwirtschaft, selbständiger Arbeit und Gewerbebetrieb. 2 . 2. Der Überschuss der Einnahmen über die Werbungskosten ( §§ 15 und 16) bei den anderen Einkunftsarten. Als gewerbliche Einkünfte (Abs. 3 Z 3) gelten stets und in vollem Umfang Einkünfte aus der Tätigkeit der offenen Gesellschaften, Kommanditgesellschaften und anderer Gesellschaft

In [29]:
# test
print(rag_answer(dataset.iloc[0]["prompt"]))

Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Beantworte die steuerrechtliche Frage. Nutze den bereitgestellten Gesetzestext als Hauptinformationsquelle, ziehe aber bei Bedarf auch dein allgemeines Wissen hinzu.

Frage:
Was ist die steuerliche Bemessungsgrundlage für die Körperschaftsteuer?

Gesetzestext:
Einkünften aus dieser Betätigung oder diesem Betrieb frühestmöglich zu verrechnen. (Anm.: Abs. 2b aufgehoben durch BGBl. I Nr. 13/2014 ) 4 . (3) Der Einkommensteuer unterliegen nur: 1 . 1. Einkünfte aus Land- und Forstwirtschaft ( § 21 ), 2 . 2. Einkünfte aus selbständiger Arbeit ( § 22 ), 3 . 3. Einkünfte aus Gewerbebetrieb ( § 23 ), 4 . 4. Einkünfte aus nichtselbständiger Arbeit ( § 25 ), 5 . 5. Einkünfte aus Kapitalvermögen ( § 27 ), 6 . 6. Einkünfte aus Vermietung und Verpachtung ( § 28 ), 7 . 7. sonstige Einkünfte im Sinne des § 29 . 5 . (4) Einkünfte im Sinne des Abs. 3 sind: 1 . 1. Der 

Antwort:


In [33]:
import pandas as pd

# load dataset with questions
dataset = pd.read_csv("dataset_clean.csv")

answers = []  # list to store generated answers

total_questions = len(dataset)  # total number of questions

for i, row in dataset.iterrows():

    question_id = row["id"]        # extract question id
    question = row["prompt"]       # extract question text

    try:
        answer = rag_answer(question)  # generate rag-based answer

        # fallback if model returns empty answer
        if answer.strip() == "":
            answer = "Keine eindeutige Antwort im Gesetzestext gefunden."

    except Exception as e:
        answer = "Fehler bei der Antwortgenerierung."  # fallback if error occurs

    answers.append({
        "id": question_id,
        "answer": answer
    })

    # print progress every 50 questions
    if (i + 1) % 50 == 0:
        print(f"{i+1}/{total_questions} Fragen beantwortet")

print("fertig")  # signal completion ✅

Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `ma

50/643 Fragen beantwortet


Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `ma

100/643 Fragen beantwortet


Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `ma

...

550/643 Fragen beantwortet


Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `ma

600/643 Fragen beantwortet


Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `ma

fertig


In [35]:
import pandas as pd

# load dataset with questions
dataset = pd.read_csv("dataset_clean.csv")

answers = []  # store generated answers
total_questions = len(dataset)

for i, row in dataset.iterrows():

    question_id = row["id"]      # question id
    question = row["prompt"]     # question text

    try:
        answer = rag_answer(question)  # generate answer via rag pipeline

        if answer.strip() == "":  # fallback if empty response
            answer = "Keine eindeutige Antwort im Gesetzestext gefunden."

    except Exception:
        answer = "Fehler bei der Antwortgenerierung."  # fallback if error

    answers.append({
        "id": question_id,
        "answer": answer
    })

    if (i + 1) % 50 == 0:  # progress update
        print(f"{i+1}/{total_questions} Fragen beantwortet")

print("fertig")

CSV gespeichert: rag_model_output.csv


In [36]:
# check
results_df.head()

,id,answer
0,CORP-TAX-001,Beantworte die steuerrechtliche Frage. Nutze d...
1,CORP-TAX-002,Beantworte die steuerrechtliche Frage. Nutze d...
2,CORP-TAX-003,Beantworte die steuerrechtliche Frage. Nutze d...
3,CORP-TAX-004,Beantworte die steuerrechtliche Frage. Nutze d...
4,CORP-TAX-005,Beantworte die steuerrechtliche Frage. Nutze d...


In [37]:
from google.colab import files
files.download("rag_model_output.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>